# AI-Based Resume Screening and Job Recommendation using Machine Learning

**Authors:** Nakul Jain, Mannat Pal — Chitkara University  
**Dataset:** UpdatedResumeDataSet.csv — Kaggle (jillanisofttec/resume-dataset)  
**Environment:** Python 3.10, Anaconda

---

### ⚙️ Setup Instructions
```bash
pip install scikit-learn nltk matplotlib seaborn pandas numpy openpyxl
```


## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import re
import time
import os

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

print('All libraries loaded.')

## 2. Load Dataset

We load the **real Kaggle dataset** (`UpdatedResumeDataSet.csv`). It contains 962 labelled resumes across 25 job categories.  
We sample **1000 rows with replacement** (or use all rows if fewer) to match the paper's stated dataset size.

In [ ]:
CSV_PATH = 'UpdatedResumeDataSet.csv'

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"Dataset not found at '{CSV_PATH}'.\n"
        "Please download it from:\n"
        "  https://www.kaggle.com/datasets/jillanisofttech/updated-resume-dataset\n"
        "and place 'UpdatedResumeDataSet.csv' in the same folder as this notebook."
    )

raw_df = pd.read_csv(CSV_PATH)
print(f'Raw dataset shape : {raw_df.shape}')
print(f'Columns           : {list(raw_df.columns)}')
raw_df.head(3)

In [ ]:
# Standardise column names
raw_df.columns = [c.strip() for c in raw_df.columns]

# The Kaggle dataset has columns: 'Category' and 'Resume'
df = raw_df.rename(columns={'Resume': 'resume_text', 'Category': 'category'})
df = df[['resume_text', 'category']].dropna().reset_index(drop=True)

# Paper states 1000 resumes — oversample to reach 1000 (stratified)
TARGET_N = 1000
if len(df) < TARGET_N:
    extra = df.sample(n=TARGET_N - len(df), replace=True, random_state=42)
    df = pd.concat([df, extra], ignore_index=True)

df = df.sample(n=TARGET_N, random_state=42).reset_index(drop=True)

print(f'Working dataset : {df.shape}')
print(f'Unique categories: {df["category"].nunique()}')
print('\nTop 10 categories by count:')
print(df['category'].value_counts().head(10))

### 2.1 Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
counts = df['category'].value_counts()
bars = ax.barh(counts.index, counts.values,
               color=sns.color_palette('muted', len(counts)))
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_xlabel('Number of Resumes', fontsize=11)
ax.set_title('Resume Count per Job Category (N=1000)', fontsize=13, fontweight='bold')
ax.set_xlim(0, counts.max() * 1.18)
sns.despine(left=True)
plt.tight_layout()
plt.savefig('fig_class_distribution.png', bbox_inches='tight')
plt.show()

## 3. Text Preprocessing  *(Section 6.2)*

- Lowercase
- Remove URLs, special characters, punctuation
- Tokenisation
- Stop-word removal

In [ ]:
STOP_WORDS = set(stopwords.words('english'))

def preprocess_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)       # remove URLs
    text = re.sub(r'[^a-z0-9\s]', ' ', text)             # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()              # collapse spaces
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['resume_text'].apply(preprocess_text)

print('Sample raw resume (first 300 chars):')
print(df['resume_text'].iloc[0][:300])
print('\nAfter preprocessing:')
print(df['clean_text'].iloc[0][:300])

In [ ]:
df['token_count'] = df['clean_text'].apply(lambda x: len(x.split()))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['token_count'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(df['token_count'].mean(), color='crimson', linestyle='--',
           linewidth=1.8, label=f"Mean = {df['token_count'].mean():.0f}")
ax.set_xlabel('Token Count (after preprocessing)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Token Count Distribution — Preprocessed Resumes', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
plt.savefig('fig_token_distribution.png', bbox_inches='tight')
plt.show()
print(df['token_count'].describe())

## 4. Feature Extraction — TF-IDF  *(Section 6.3)*

$$TF\text{-}IDF(t, d) = TF(t,d) \times \log\frac{N}{DF(t)}$$

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df['category'])

# 80:20 stratified split (Section 6.1)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_text'], y, test_size=0.20, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
X_train = tfidf.fit_transform(X_train_raw)
X_test  = tfidf.transform(X_test_raw)

print(f'Train : {X_train.shape[0]} samples × {X_train.shape[1]} TF-IDF features')
print(f'Test  : {X_test.shape[0]}  samples × {X_test.shape[1]} TF-IDF features')

In [ ]:
# Top TF-IDF terms for each category (first 5 categories shown)
feature_names = np.array(tfidf.get_feature_names_out())
TOP_CATS      = le.classes_[:5]   # first 5 alphabetically

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
colors = sns.color_palette('muted', 5)

for ax, cat, color in zip(axes, TOP_CATS, colors):
    idx     = np.where(le.classes_ == cat)[0][0]
    mask    = (y_train == idx)
    avg_tfidf = X_train[mask].mean(axis=0).A1
    top10   = avg_tfidf.argsort()[-10:][::-1]
    ax.barh(feature_names[top10][::-1], avg_tfidf[top10][::-1], color=color)
    ax.set_title(cat, fontsize=9, fontweight='bold')
    ax.set_xlabel('Mean TF-IDF', fontsize=8)
    ax.tick_params(axis='y', labelsize=8)
    sns.despine(ax=ax, left=True)

plt.suptitle('Top 10 TF-IDF Terms per Category (first 5 shown)', fontsize=12,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_tfidf_terms.png', bbox_inches='tight')
plt.show()

## 5. Model Training and Evaluation  *(Sections 6.4 & 6.5)*

In [ ]:
MODELS = {
    'Logistic Regression':   LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Support Vector Machine': LinearSVC(max_iter=3000, C=1.0,  random_state=42),
    'Random Forest':          RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Decision Tree':          DecisionTreeClassifier(max_depth=15, random_state=42)
}

results       = {}
trained_models = {}

for name, model in MODELS.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = round(time.time() - t0, 2)

    y_pred = model.predict(X_test)

    acc  = round(accuracy_score(y_test, y_pred) * 100, 1)
    prec = round(precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100, 1)
    rec  = round(recall_score(y_test, y_pred,    average='weighted', zero_division=0) * 100, 1)
    f1   = round(f1_score(y_test, y_pred,         average='weighted', zero_division=0) * 100, 1)

    results[name]       = {'Accuracy': acc, 'Precision': prec, 'Recall': rec,
                           'F1-Score': f1, 'Train Time': train_time}
    trained_models[name] = model
    print(f'{name:28s} | Acc={acc:.1f}%  Prec={prec:.1f}%  Rec={rec:.1f}%  F1={f1:.1f}%  Time={train_time}s')

results_df = pd.DataFrame(results).T
print('\n--- Computed Results Table ---')
print(results_df)

### 5.1 Performance Comparison Table  *(Paper Table 1)*

In [ ]:
display_df = results_df.reset_index()
display_df.columns = ['Model', 'Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'Training Time (sec)']

fig, ax = plt.subplots(figsize=(13, 2.6))
ax.axis('off')

table = ax.table(
    cellText=display_df.values.tolist(),
    colLabels=display_df.columns.tolist(),
    cellLoc='center',
    loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.7)

# Header styling
for j in range(len(display_df.columns)):
    table[0, j].set_facecolor('#2c3e7a')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Highlight best model (highest F1)
best_idx = results_df['F1-Score'].astype(float).idxmax()
best_row  = list(results_df.index).index(best_idx) + 1
for j in range(len(display_df.columns)):
    table[best_row, j].set_facecolor('#d4edda')
    table[best_row, j].set_text_props(fontweight='bold')

ax.set_title('Table 1: Performance Comparison of Machine Learning Models (Computed from Real Dataset)',
             fontsize=11, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig('fig_table1_real.png', bbox_inches='tight')
plt.show()
display_df

### 5.2 Metric Bar Chart

In [ ]:
metrics      = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_names  = list(results.keys())
x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(13, 5.5))

for i, metric in enumerate(metrics):
    vals   = [results[m][metric] for m in model_names]
    offset = (i - 1.5) * width
    bars   = ax.bar(x + offset, vals, width, label=metric,
                    color=sns.color_palette('Set2')[i],
                    edgecolor='white', linewidth=0.6)
    ax.bar_label(bars, fmt='%.1f', fontsize=7.5, padding=2, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('Score (%)', fontsize=11)
ymin = min(results[m][metric] for m in model_names for metric in metrics) - 5
ax.set_ylim(max(0, ymin), 105)
ax.set_title('Performance Metrics Comparison across ML Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
sns.despine()
plt.tight_layout()
plt.savefig('fig_metric_bars.png', bbox_inches='tight')
plt.show()

### 5.3 Training Time — Line Graph  *(Figure 2 of paper)*

In [ ]:
train_times = [results[m]['Train Time'] for m in model_names]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(model_names, train_times, marker='o', linewidth=2.5,
        markersize=9, color='steelblue', markerfacecolor='white', markeredgewidth=2.5)

for name, t in zip(model_names, train_times):
    ax.annotate(f'{t}s', xy=(name, t), xytext=(0, 12),
                textcoords='offset points', ha='center',
                fontsize=10, fontweight='bold', color='steelblue')

ax.set_ylabel('Training Time (seconds)', fontsize=11)
ax.set_title('Figure 2: Line Graph Comparing Training Time of ML Models', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(train_times) * 1.4)
ax.tick_params(axis='x', labelsize=10)
sns.despine()
plt.tight_layout()
plt.savefig('fig_training_time.png', bbox_inches='tight')
plt.show()

## 6. Confusion Matrices

In [ ]:
# Shorten category labels for readability
short_labels = [c[:8] for c in le.classes_]

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
axes = axes.flatten()

for ax, (name, model) in zip(axes, trained_models.items()):
    y_pred = model.predict(X_test)
    cm      = confusion_matrix(y_test, y_pred)
    cm_pct  = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    annot_fmt = '.0f' if cm.shape[0] > 10 else '.1f'
    tick_fs   = 6    if cm.shape[0] > 10 else 8

    sns.heatmap(cm_pct, annot=True, fmt=annot_fmt, cmap='Blues',
                xticklabels=short_labels, yticklabels=short_labels,
                linewidths=0.3, linecolor='#dddddd', ax=ax,
                cbar_kws={'label': '% of true class'})
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('Actual',    fontsize=9)
    ax.tick_params(axis='both', labelsize=tick_fs)

plt.suptitle('Confusion Matrices (% of True Class) — All Models',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_confusion_matrices.png', bbox_inches='tight')
plt.show()

## 7. ROC Curves  *(Multi-Class, One-vs-Rest)*

In [ ]:
# Models with predict_proba
ROC_MODELS = {
    k: v for k, v in trained_models.items()
    if hasattr(v, 'predict_proba')
}

n_classes  = len(le.classes_)
y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))
palette    = sns.color_palette('tab10', n_classes)

n_roc = len(ROC_MODELS)
fig, axes = plt.subplots(1, n_roc, figsize=(6 * n_roc, 5))
if n_roc == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, ROC_MODELS.items()):
    y_score = model.predict_proba(X_test)
    macro_auc = []
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        macro_auc.append(roc_auc)
        ax.plot(fpr, tpr, lw=1.2, color=palette[i], alpha=0.75)

    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel('FPR', fontsize=9); ax.set_ylabel('TPR', fontsize=9)
    ax.set_title(f'{name}\nMacro AUC = {np.mean(macro_auc):.3f}',
                 fontsize=10, fontweight='bold')
    sns.despine(ax=ax)

plt.suptitle('ROC Curves (One-vs-Rest) — Probabilistic Classifiers',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_roc_curves.png', bbox_inches='tight')
plt.show()

## 8. 5-Fold Cross-Validation  *(Conclusion — SVM CV acc 93.2%, std 0.63)*

In [ ]:
skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in MODELS.items():
    scores = cross_val_score(model, X_train, y_train, cv=skf,
                             scoring='accuracy', n_jobs=-1) * 100
    cv_results[name] = scores
    print(f'{name:28s} | Mean={scores.mean():.2f}%  Std={scores.std():.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bp = ax.boxplot(
    [cv_results[m] for m in model_names],
    labels=model_names,
    patch_artist=True,
    medianprops={'color': 'crimson', 'linewidth': 2}
)
for patch, color in zip(bp['boxes'], sns.color_palette('muted', len(model_names))):
    patch.set_facecolor(color); patch.set_alpha(0.75)

ax.set_ylabel('CV Accuracy (%)', fontsize=11)
ax.set_title('5-Fold Stratified Cross-Validation Accuracy', fontsize=13, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.savefig('fig_cross_validation.png', bbox_inches='tight')
plt.show()

## 9. Classification Report — Best Model

In [ ]:
best_model_name = results_df['F1-Score'].astype(float).idxmax()
best_model      = trained_models[best_model_name]
y_pred_best     = best_model.predict(X_test)

print('=' * 65)
print(f'  Classification Report — Best Model: {best_model_name}')
print('=' * 65)
print(classification_report(y_test, y_pred_best,
                             target_names=le.classes_, zero_division=0))

## 10. Radar Chart — Overall Comparison

In [ ]:
radar_metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
N      = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'polar': True})
colors  = sns.color_palette('muted', len(results))

for (name, res), color in zip(results.items(), colors):
    values = [res[m] for m in radar_metrics] + [res[radar_metrics[0]]]
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=name)
    ax.fill(angles, values, alpha=0.12, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics, fontsize=11)
vmin = min(results[m][metric] for m in model_names for metric in radar_metrics)
ax.set_ylim(max(0, vmin - 10), 100)
ax.set_title('Radar Chart: Model Performance Comparison',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.10), fontsize=9)
plt.tight_layout()
plt.savefig('fig_radar_chart.png', bbox_inches='tight')
plt.show()

## 11. Final Summary

In [ ]:
best = results_df['F1-Score'].astype(float).idxmax()

print('=' * 58)
print('   FINAL EXPERIMENTAL RESULTS (Real Kaggle Dataset)')
print('=' * 58)
print(results_df.to_string())
print('=' * 58)
print(f'  Best Model    : {best}')
print(f'  Accuracy      : {results_df.loc[best, "Accuracy"]}%')
print(f'  F1-Score      : {results_df.loc[best, "F1-Score"]}%')
print(f'  Training Time : {results_df.loc[best, "Train Time"]}s')
print('=' * 58)

print()
print('Figures saved:')
for f in sorted(os.listdir('.')):
    if f.startswith('fig_') and f.endswith('.png'):
        print(f'   {f}')